In [1]:
import numpy as np
import matplotlib.pyplot as plt
import nifty8 as ift
from nipype.interfaces.cmtk.cmtk import save_fibers

from phase_II.utils.helpers import Stress, visualize_stress_updated, usual_plot_updated
from data.style_components.matplotlib_style import *
from phase_I.utils.config_jupyter_notebooks import *
%matplotlib tk

nrt_strain_values = np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_strain_values.txt") * 1e19
nrt_time_values = np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_time_values.txt") - zero_time

Important variables: 
		signal_strip_time, signal_strip_strain 
		signal_strip_strain_tapered
		strain
		time_domain_strip
		N


In [2]:
def generate_white_noise_stress_matrices(number_of_matrices, time_domain, supress_print=False):
    if number_of_matrices == 1:
            real_space_white_noise = ift.from_random(time_domain)
            stress, t_dual, f_dual = Stress(real_space_white_noise, supress_print=supress_print)
            return stress, t_dual, f_dual

    S_mat_collection = []
    for i in range(number_of_matrices):
            print(f"Calcuting white noise stress matrix, iteration {i} out of {number_of_matrices}")
            real_space_white_noise = ift.from_random(time_domain)
            stress, _, _ = Stress(real_space_white_noise, supress_print=supress_print)
            S_mat_collection.append(np.array(stress))

    print("Done, wrapping result in numpy array")
    return np.array(S_mat_collection)

dt = nrt_time_values[1]-nrt_time_values[0]
time_domain = ift.RGSpace(shape=(len(nrt_time_values),), distances=dt)
white_noise_stress, t_dual, f_dual = generate_white_noise_stress_matrices(number_of_matrices=1, time_domain=time_domain)

pure_white_noise = 600*np.random.standard_normal(white_noise_stress.shape)+10



Beginning stress calculation...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Fourier-Transforming columns of Phi matrix
	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (7.101603990408326e-17) 


In [8]:
# visualize_stress_updated(white_noise_stress, rows=f_dual, cols=t_dual, smooth=False)
visualize_stress_updated(pure_white_noise, rows=f_dual, cols=t_dual, smooth=True)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


### Get the numerical relativity template

In [3]:
L = np.max(nrt_time_values) - np.min(nrt_time_values)
n = len(nrt_time_values)
time_space = ift.RGSpace((n,), distances=L/n)

nrt_field = ift.Field(ift.DomainTuple.make(time_space), val=nrt_strain_values)

_ = plt.figure(figsize=(10,6))
plt.plot(nrt_time_values, nrt_field.val)
usual_plot_updated(xlim=(16, 16.75), title=r"$\xi(t)$", save_fig=False)

#### Get and plot its empirical power spectrum

In [4]:
r_dom = nrt_field.domain
h_dom = r_dom[0].get_default_codomain()
F = ift.FFTOperator(domain=r_dom)
harmonic_nrt_field = F(nrt_field)

empirical_ps_nrt = ift.power_analyze(harmonic_nrt_field)

In [5]:
_ = plt.figure()
plt.plot(h_dom.get_unique_k_lengths(), empirical_ps_nrt.val, label="abs. square of fourier transform")
plt.xlabel('Unique k lengths')
plt.ylabel("Power")
plt.loglog()
plt.show()

#### Samples based on this power spectrum

In [ ]:
def expand_rfft(f_unique, N):
    # work with unique k's and broadcast to full k's using this function.
    return np.concatenate([f_unique, f_unique[-2:0:-1].conj()]) if N % 2 == 0 else np.concatenate([f_unique, f_unique[-1:0:-1].conj()])

In [ ]:
num = 1

FFT = lambda p: np.fft.fft(p, n=len(nrt_time_values), norm="ortho")
iFFT = lambda p: np.fft.ifft(p, n=len(nrt_time_values), norm="ortho")

full_field_power_spectrum = expand_rfft(empirical_ps_nrt.val, N=len(nrt_time_values))
field_samples = []

for _ in range(num):
    xi = np.random.standard_normal(len(nrt_time_values))
    harmonic_xi = FFT(xi)
    tmp = full_field_power_spectrum * harmonic_xi
    field_samples.append(iFFT(tmp))

In [ ]:
_ = plt.figure(figsize=(10,6))

for sl in field_samples:
    plt.plot(nrt_time_values, sl.real)

# plt.plot(nrt_time_values, nrt_field.val, label="Num. rel. template")
usual_plot(title="Num. rel. template and samples from its pow spec")

#### Stress of the field

In [6]:
S_mat, t_dual, f = Stress(nrt_field)


Beginning stress calculation...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Fourier-Transforming columns of Phi matrix
	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (1.4803592132673052e-22) 


In [9]:
# visualize_stress_updated(S_mat, rows=f, cols=t_dual+min(nrt_time_values), xlim=(16.2, 16.5), ylim=(-500,500), save_fig=False)
visualize_stress_updated(S_mat.T, rows=f, cols=t_dual+min(nrt_time_values),  save_fig=False)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


## Try to invert the Wigner function

In [ ]:
t_vol = r_dom[0].scalar_dvol
N = r_dom[0].shape[0]
h_vol = h_dom.scalar_dvol*N

FFT_1 = ift.FFTOperator(domain=(h_dom, r_dom[0]), space=1) * (1/t_vol)

S_mat_field = ift.Field(domain=ift.DomainTuple.make((h_dom, r_dom[0])), val=S_mat)

In [ ]:
Sigma_f_q = FFT_1(S_mat_field).val

In [ ]:
plt.matshow(np.abs(Sigma_f_q))

In [ ]:
# Get Sigma (q/2, q)
k_half = 0.5 * f.copy()
xi_0 = harmonic_nrt_field.val[0]

### Approach: Just pick out every second wavevector => Introduces shift and stretch in time
k_half_in_f = np.isin(k_half, f)
masked_k_half = k_half[k_half_in_f]

valid_columns = np.where(k_half_in_f)[0]

print(valid_columns)

valid_rows = np.array([np.where(f == kh)[0][0] for kh in masked_k_half])

# reconstructed_xi_tilde_phase = np.zeros(len(f), dtype=complex)
# to_insert = Sigma_f_q[valid_rows, valid_columns]
# reconstructed_xi_tilde_phase[valid_columns] = to_insert

reconstructed_xi_tilde_phase = Sigma_f_q[valid_rows, valid_columns]
reconstructed_xi_tilde_wo_phase = reconstructed_xi_tilde_phase/xi_0.conj()

delta_k = masked_k_half[1] - masked_k_half[0]
K = len(masked_k_half)
dual_coarse_time = np.arange(K) / (K * delta_k)
dual_fine_time = np.arange(len(nrt_time_values)) / (len(nrt_time_values) * (f[1]-f[0]))

coarser_h_dom = ift.RGSpace(harmonic=True, shape=(len(masked_k_half), ), distances=delta_k)

h_vol_coarser = coarser_h_dom.scalar_dvol*len(masked_k_half)
FFT_2_coarse = ift.FFTOperator(domain=coarser_h_dom) * (1/h_vol_coarser)

FFT_2_fine = ift.FFTOperator(domain=h_dom) * (1/h_vol)

reconstructed_xi_tilde_phase_field =  ift.Field(ift.DomainTuple.make(coarser_h_dom), val=reconstructed_xi_tilde_phase)
reconstructed_xi_tilde_wo_phase_field =  ift.Field(ift.DomainTuple.make(coarser_h_dom), val=reconstructed_xi_tilde_wo_phase)

### Approach: Linear interpolate

# preallocate
# reconstructed_xi_tilde_phase = np.empty_like(k_half, dtype=complex)

# interpolate each column of Sigma_f_q along the first axis
# for j in range(Sigma_f_q.shape[1]):
#     reconstructed_xi_tilde_phase[j] = np.interp(
#         np.fft.fftshift(k_half)[j],       # target x
#         np.fft.fftshift(f),               # original x, need to be ordered monotonically => np.fft.fftshift
#         np.fft.fftshift(Sigma_f_q[:, j].real)  # y-values at original x
#     )
#     +1j * np.interp(
#         np.fft.fftshift(k_half)[j],
#         np.fft.fftshift(f),
#         np.fft.fftshift(Sigma_f_q[:, j].imag)
#     )
#
# reconstructed_xi_tilde_phase = np.fft.ifftshift(reconstructed_xi_tilde_phase)
# reconstructed_xi_tilde_wo_phase = reconstructed_xi_tilde_phase/xi_0.conj()  # inverse shift to get ready for FFT!
#
# FFT_2 = ift.FFTOperator(domain=h_dom) * (1/h_vol)
#
# reconstructed_xi_tilde_phase_field =  ift.Field(ift.DomainTuple.make(h_dom), val=reconstructed_xi_tilde_phase)
# reconstructed_xi_tilde_wo_phase_field =  ift.Field(ift.DomainTuple.make(h_dom), val=reconstructed_xi_tilde_wo_phase)


In [ ]:
recovered_xi = FFT_2_coarse(reconstructed_xi_tilde_wo_phase_field).val
recovered_xi_normed = recovered_xi/np.max(recovered_xi) * np.max(nrt_field.val)

recovered_xi_up_to_phase = FFT_2_coarse(reconstructed_xi_tilde_phase_field).val
recovered_xi_up_to_phase_normed = recovered_xi_up_to_phase/np.max(recovered_xi_up_to_phase) * np.max(nrt_field.val)

In [ ]:
# plt.plot(np.arange(len(recovered_xi_normed)), recovered_xi_up_to_phase_normed, label="Up to global phase")
plt.plot(np.arange(len(recovered_xi_normed)), recovered_xi_normed, label="Fourier back transform of inverse wigner mat (scaled f.v.p.!)")
plt.plot(np.arange(len(nrt_field.val))-4100, nrt_field.val, label="Original input field (shifted to the left)")
usual_plot(xl="Index")